# Feature Engineering and Transformation

Task 1 evidence notebook for geolocation integration, time/velocity features, scaling, one-hot encoding, and training-only imbalance handling.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))

from src.preprocessing import clean_fraud_data, clean_creditcard_data, load_csv, build_preprocessor, apply_training_resampling
from src.feature_engineering import add_geolocation_features, add_ecommerce_time_velocity_features, add_creditcard_features, country_fraud_summary

## Load and clean both datasets

Cleaning covers missing values, duplicate rows, and type corrections for `fraud_data.csv` and `creditcard.csv`.

In [ ]:
raw = ROOT / 'data' / 'raw'
fraud_raw = load_csv(raw / 'fraud_data.csv')
credit_raw = load_csv(raw / 'creditcard.csv')
ip_ranges = load_csv(raw / 'IpAddress_to_Country.csv')

fraud_clean, fraud_quality = clean_fraud_data(fraud_raw)
credit_clean, credit_quality = clean_creditcard_data(credit_raw)
fraud_quality, credit_quality

## Geolocation and feature engineering

IP values are converted to integers, matched against country ranges, then used for country-level fraud analysis. Time and velocity features include `time_since_signup`, `hour_of_day`, `day_of_week`, and `transaction_velocity`.

In [ ]:
fraud_geo = add_geolocation_features(fraud_clean, ip_ranges)
fraud_features = add_ecommerce_time_velocity_features(fraud_geo)
credit_features = add_creditcard_features(credit_clean)

country_fraud_summary(fraud_features).head(10)

## Scaling, encoding, and imbalance handling

The transformation pipeline uses `StandardScaler` for numeric fields and `OneHotEncoder` for categorical fields. SMOTE is applied after the train/test split, so the test set remains untouched.

In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = ['purchase_value', 'age', 'time_since_signup', 'hour_of_day', 'day_of_week', 'transaction_velocity', 'device_transaction_count', 'ip_transaction_count']
categorical_features = ['source', 'browser', 'sex', 'country']

X = fraud_features[numeric_features + categorical_features]
y = fraud_features['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = build_preprocessor(numeric_features, categorical_features)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

X_train_resampled, y_train_resampled = apply_training_resampling(X_train_prepared, y_train, method='smote')
y_train.value_counts(normalize=True), y_train_resampled.value_counts(normalize=True)